# Days 12-14: Multimodal Reranker + Ablation + Final Demo

**Goal**:
- Run the full multimodal reranker using all 6 signals
- Produce ablation table comparing all 4 retrieval modes
- Build a final demo showing explainable results
- Save all results to Drive

**4 modes compared**:
1. Visual only
2. Visual + Metadata
3. Visual + Context (caption/geo/entity)
4. Visual + Full Context + Topic Reranking

**Done checkpoint**:
- Ablation table shows measurable improvement across modes
- Demo shows explainability cards per result
- `results/ablation.csv` updated with all 4 modes

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/PinterestDemo'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
!pip install -q torch torchvision transformers faiss-cpu Pillow pandas numpy tqdm pyyaml scipy scikit-learn
print('Done.')

## 2. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE = Path(PROJECT_DIR)
config['paths']['metadata_csv']    = str(BASE / 'data/metadata/images.csv')
config['paths']['embeddings_path'] = str(BASE / 'data/embeddings/embeddings.npy')
config['paths']['id_map_path']     = str(BASE / 'data/embeddings/id_map.json')
config['paths']['index_path']      = str(BASE / 'data/embeddings/faiss.index')
config['paths']['eval_dir']        = str(BASE / 'data/eval')
config['paths']['results_dir']     = str(BASE / 'results')
config['model']['device']          = 'cuda'

Path(config['paths']['results_dir']).mkdir(parents=True, exist_ok=True)
print('Config ready.')

## 3. Initialize All Components

In [ ]:
from src.retrieval.search import Searcher
from src.reranking.metadata_reranker import MetadataReranker
from src.reranking.multimodal_reranker import MultimodalReranker

searcher = Searcher(config)
metadata_reranker = MetadataReranker(config)
multimodal_reranker = MultimodalReranker(config, searcher.encoder)

# Try loading topic index for multimodal reranker
topic_index_path   = str(BASE / 'data/embeddings/topic_index.faiss')
topic_records_path = str(BASE / 'data/embeddings/topic_records.json')

if Path(topic_index_path).exists():
    from src.enrichment.topic_index import TopicIndex
    ti = TopicIndex()
    ti.load(index_path=topic_index_path, records_path=topic_records_path)
    multimodal_reranker.topic_index = ti
    print(f'Topic index loaded: {ti.index.ntotal} topics')
else:
    print('Topic index not found. Run Day 11 notebook to enable trend signals.')

print(f'\nAll components ready.')
print(f'  Searcher index:  {searcher.index.index.ntotal:,} vectors')

## 4. Run Full Ablation Evaluation

In [ ]:
from src.evaluation.eval_set import build_eval_set, load_eval_set
from src.evaluation.metrics import compute_metrics
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

k_values = config['evaluation']['k_values']
candidate_pool = config['retrieval']['candidate_pool']

# Load or build eval set
eval_path = Path(config['paths']['eval_dir']) / 'eval_queries.csv'
if not eval_path.exists():
    build_eval_set(config, n_queries=200)
eval_queries = load_eval_set(config)
print(f'Eval set: {len(eval_queries)} queries')

meta_df = pd.read_csv(config['paths']['metadata_csv'])
meta_df['image_id'] = meta_df['image_id'].astype(int)
meta_index = meta_df.set_index('image_id')

# Collect results for all 4 modes
mode_evals = {
    'visual_only': [],
    'metadata_reranked': [],
    'context_reranked': [],
    'full_multimodal': [],
}

for q in tqdm(eval_queries, desc='Evaluating all modes'):
    qid = q['query_image_id']
    relevant = q['relevant']

    if qid not in meta_index.index:
        continue
    query_path = meta_index.loc[qid].get('path')
    if not query_path or not Path(query_path).exists():
        continue

    try:
        raw = searcher.search(query_path, k=candidate_pool)
    except Exception:
        continue

    query_meta = meta_index.loc[qid].to_dict()
    k_max = max(k_values)

    # Mode 1: visual only
    mode_evals['visual_only'].append({
        'retrieved': [r['image_id'] for r in raw[:k_max]], 'relevant': relevant
    })

    # Mode 2: metadata reranked
    reranked_meta = metadata_reranker.rerank(raw, query_meta=query_meta, top_k=k_max)
    mode_evals['metadata_reranked'].append({
        'retrieved': [r['image_id'] for r in reranked_meta], 'relevant': relevant
    })

    # Mode 3: context reranked (caption + geo, no topic)
    temp_reranker = MultimodalReranker(config, searcher.encoder)
    temp_reranker.topic_index = None  # disable topic for this mode
    temp_reranker.w_topic = 0.0
    temp_reranker.w_visual = 0.60
    reranked_ctx = temp_reranker.rerank(raw, query_meta=query_meta, top_k=k_max)
    mode_evals['context_reranked'].append({
        'retrieved': [r['image_id'] for r in reranked_ctx], 'relevant': relevant
    })

    # Mode 4: full multimodal
    reranked_full = multimodal_reranker.rerank(raw, query_meta=query_meta, top_k=k_max)
    mode_evals['full_multimodal'].append({
        'retrieved': [r['image_id'] for r in reranked_full], 'relevant': relevant
    })

print(f'\nEvaluation done for {len(mode_evals["visual_only"])} queries.')

## 5. Compute + Display Ablation Table

In [ ]:
from src.evaluation.metrics import compute_metrics, print_metrics_table
import pandas as pd

all_metrics = {}
for mode_name, evals in mode_evals.items():
    if evals:
        all_metrics[mode_name] = compute_metrics(evals, k_values=k_values)

# Display as table
rows = []
for mode_name, metrics in all_metrics.items():
    row = {'Mode': mode_name}
    row.update({k: f'{v:.4f}' for k, v in sorted(metrics.items())})
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.set_index('Mode', inplace=True)
print('ABLATION RESULTS:')
display(results_df)

## 6. Save Ablation CSV

In [ ]:
import csv
from pathlib import Path

ablation_csv = Path(config['paths']['results_dir']) / 'ablation.csv'
fieldnames = ['mode'] + sorted(list(next(iter(all_metrics.values())).keys()))

with open(ablation_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for mode_name, metrics in all_metrics.items():
        writer.writerow({'mode': mode_name, **metrics})

print(f'Ablation saved to {ablation_csv}')

## 7. Visualize Ablation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics_to_plot = ['Recall@10', 'mAP', 'nDCG@10']
modes = list(all_metrics.keys())
x = np.arange(len(metrics_to_plot))
width = 0.18
colors = ['steelblue', 'coral', 'seagreen', 'orchid']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (mode, metrics) in enumerate(all_metrics.items()):
    vals = [metrics.get(m, 0) for m in metrics_to_plot]
    bars = ax.bar(x + i*width - (len(modes)-1)*width/2, vals, width,
                  label=mode.replace('_', ' ').title(), color=colors[i % len(colors)])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Ablation: 4 Retrieval Modes Compared', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/ablation_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/ablation_chart.png')

## 8. Final Demo with Explainability Cards

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import pandas as pd

def final_demo(query_path, top_k=8):
    """Show visual-only vs full context side-by-side with explanation."""
    meta_row = meta_df[meta_df['path'] == query_path].iloc[0].to_dict() if not meta_df[meta_df['path'] == query_path].empty else {}
    query_caption = str(meta_row.get('caption', '') or '')

    raw = searcher.search(query_path, k=config['retrieval']['candidate_pool'])
    visual_results = raw[:top_k]
    full_results = multimodal_reranker.rerank(raw, query_meta=meta_row, query_caption=query_caption, top_k=top_k)

    fig = plt.figure(figsize=(20, 12))
    cols = top_k // 2 + 1

    # Query image (top left)
    ax_q = fig.add_subplot(3, cols, 1)
    ax_q.imshow(Image.open(query_path).resize((224, 224)))
    ax_q.set_title(f'QUERY\n{meta_row.get("category","")[:15]}\n{query_caption[:40]}',
                   color='royalblue', fontsize=8, fontweight='bold')
    ax_q.axis('off')

    # Visual-only row
    for i, r in enumerate(visual_results[:cols-1]):
        ax = fig.add_subplot(3, cols, cols + i + 1)
        if r.get('path') and Path(r['path']).exists():
            ax.imshow(Image.open(r['path']).resize((224, 224)))
        title = f'#{i+1} {r.get("category","")[:10]}\nscore={r["score"]:.3f}'
        ax.set_title(title, fontsize=7, color='steelblue')
        ax.axis('off')

    # Full context row
    for i, r in enumerate(full_results[:cols-1]):
        ax = fig.add_subplot(3, cols, 2*cols + i + 1)
        if r.get('path') and Path(r['path']).exists():
            ax.imshow(Image.open(r['path']).resize((224, 224)))
        exp = r.get('explanation', {})
        vis = exp.get('visual_similarity', 0)
        ctx = exp.get('caption_similarity', 0) + exp.get('metadata_match', 0)
        title = f'#{i+1} {r.get("category","")[:10]}\nvis={vis:.2f} ctx={ctx:.2f}'
        ax.set_title(title, fontsize=7, color='seagreen')
        ax.axis('off')

    # Row labels
    ax_label1 = fig.add_subplot(3, cols, cols + 1)
    fig.text(0.01, 0.55, 'VISUAL\nONLY', va='center', ha='left', fontsize=10,
             color='steelblue', fontweight='bold', rotation=90)
    fig.text(0.01, 0.25, 'FULL\nCONTEXT', va='center', ha='left', fontsize=10,
             color='seagreen', fontweight='bold', rotation=90)

    plt.suptitle('Visual Search: Visual-Only vs Context-Aware', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run demo for 3 different queries
for _, row in meta_df.sample(3).iterrows():
    if row.get('path') and Path(row['path']).exists():
        final_demo(row['path'])

## 9. Export Qualitative Examples

In [ ]:
# Save 10 qualitative comparison screenshots
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

qual_dir = Path(PROJECT_DIR) / 'results' / 'qualitative_examples'
qual_dir.mkdir(parents=True, exist_ok=True)

for idx, (_, row) in enumerate(meta_df.sample(10).iterrows()):
    if not (row.get('path') and Path(row['path']).exists()):
        continue

    query_path = row['path']
    raw = searcher.search(query_path, k=config['retrieval']['candidate_pool'])
    visual = raw[:6]
    full = multimodal_reranker.rerank(raw, query_meta=row.to_dict(), top_k=6)

    fig, axes = plt.subplots(2, 7, figsize=(21, 6))

    # Query
    for ax_row in axes:
        ax_row[0].imshow(Image.open(query_path).resize((224,224)))
        ax_row[0].set_title('Query', fontweight='bold', fontsize=8)
        ax_row[0].axis('off')

    for i, r in enumerate(visual[:6]):
        axes[0][i+1].imshow(Image.open(r['path']).resize((224,224)) if r.get('path') and Path(r['path']).exists() else Image.new('RGB', (224,224)))
        axes[0][i+1].set_title(f'{r.get("category","")[:10]}\n{r["score"]:.3f}', fontsize=7, color='steelblue')
        axes[0][i+1].axis('off')

    for i, r in enumerate(full[:6]):
        axes[1][i+1].imshow(Image.open(r['path']).resize((224,224)) if r.get('path') and Path(r['path']).exists() else Image.new('RGB', (224,224)))
        axes[1][i+1].set_title(f'{r.get("category","")[:10]}\n{r["score"]:.3f}', fontsize=7, color='seagreen')
        axes[1][i+1].axis('off')

    axes[0][0].set_ylabel('Visual Only', fontsize=9, color='steelblue')
    axes[1][0].set_ylabel('Full Context', fontsize=9, color='seagreen')

    plt.suptitle(f'Example {idx+1}: {row.get("product_name","")[:40]}', fontsize=11)
    plt.tight_layout()

    save_path = qual_dir / f'example_{idx+1:02d}.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()

print(f'Qualitative examples saved to {qual_dir}')
print(f'Files: {list(qual_dir.glob("*.png"))[:5]}')

## ✅ Days 12-14 Done Checkpoint

In [ ]:
from pathlib import Path
import pandas as pd

ablation_csv = Path(config['paths']['results_dir']) / 'ablation.csv'
assert ablation_csv.exists(), 'ablation.csv missing'
df_results = pd.read_csv(ablation_csv)
assert len(df_results) >= 2, 'Need at least 2 modes in ablation'

qual_dir = Path(PROJECT_DIR) / 'results' / 'qualitative_examples'
n_examples = len(list(qual_dir.glob('*.png')))

print('Days 12-14 COMPLETE')
print(f'  Ablation modes:        {len(df_results)}')
print(f'  Qualitative examples:  {n_examples}')
print(f'\nFinal ablation results:')
display(df_results.set_index('mode')[['Recall@10', 'mAP', 'nDCG@10']])
print(f'\nProject is complete. Next step: push to GitHub and share!')